In [1]:
%pip install -q numpy pandas matplotlib seaborn scikit-learn opencv-python-headless scikit-image joblib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("dataset")           # thư mục chứa metadata.csv và ảnh
OUT_DIR = Path("processed_images")   # thư mục lưu ảnh sau khi xử lý
IMG_SIZE = 128                       # kích thước ảnh sau khi resize

print("Import thư viện thành công!")

Import thư viện thành công!


PHẦN 1: TIỀN XỬ LÝ DỮ LIỆU BẢNG

In [3]:
df = pd.read_csv(DATA_DIR / "metadata.csv")

print("5 DÒNG ĐẦU TIÊN:")
print(df.head())

print("\nKÍCH THƯỚC DỮ LIỆU:")
print("Số dòng:", df.shape[0])
print("Số cột:", df.shape[1])

print("\nTÊN CÁC CỘT:")
print(df.columns.tolist())

print("\nKIỂU DỮ LIỆU BAN ĐẦU:")
for col in df.columns:
    print(col, ":", df[col].dtype)

5 DÒNG ĐẦU TIÊN:
  patient_id  lesion_id  smoke  drink background_father background_mother  \
0   PAT_1516       1765    NaN    NaN               NaN               NaN   
1     PAT_46        881  False  False         POMERANIA         POMERANIA   
2   PAT_1545       1867    NaN    NaN               NaN               NaN   
3   PAT_1989       4061    NaN    NaN               NaN               NaN   
4    PAT_684       1302  False   True         POMERANIA         POMERANIA   

   age pesticide  gender skin_cancer_history  ... diameter_2 diagnostic  \
0    8       NaN     NaN                 NaN  ...        NaN        NEV   
1   55     False  FEMALE                True  ...        5.0        BCC   
2   77       NaN     NaN                 NaN  ...        NaN        ACK   
3   75       NaN     NaN                 NaN  ...        NaN        ACK   
4   79     False    MALE                True  ...        5.0        BCC   

    itch   grew   hurt  changed  bleed elevation                 img_

In [4]:
print("a) Dữ liệu thiếu:")

missing_count = df.isnull().sum()
missing_percent = (df.isnull().mean() * 100).round(2)

missing_table = pd.DataFrame({
    "Số lượng thiếu": missing_count,
    "Tỷ lệ (%)": missing_percent
})

# Chỉ in các cột có dữ liệu thiếu
print(missing_table[missing_table["Số lượng thiếu"] > 0])

print("\nb) Số dòng trùng:", df.duplicated().sum())

a) Dữ liệu thiếu:
                     Số lượng thiếu  Tỷ lệ (%)
smoke                           804      34.99
drink                           804      34.99
background_father               818      35.60
background_mother               822      35.77
pesticide                       804      34.99
gender                          804      34.99
skin_cancer_history             804      34.99
cancer_history                  804      34.99
has_piped_water                 804      34.99
has_sewage_system               804      34.99
fitspatrick                     804      34.99
diameter_1                      804      34.99
diameter_2                      804      34.99

b) Số dòng trùng: 0


In [ ]:
#KIỂM TRA GIÁ TRỊ CỦA CÁC CỘT
# Các cột mã định danh: không cần xem giá trị
id_columns = ["patient_id", "lesion_id", "img_id"]

# Các cột dạng số
numeric_columns = ["age", "fitspatrick", "diameter_1", "diameter_2"]

print("GIÁ TRỊ CỦA CÁC CỘT:")

for col in df.columns:
    if col in id_columns:
        continue

    print("\n---", col, "---")

    if col in numeric_columns:
        print(df[col].describe().round(2))
    else:
        print(df[col].value_counts(dropna=False))

GIÁ TRỊ CỦA CÁC CỘT:

--- smoke ---
smoke
False    1292
NaN       804
True      202
Name: count, dtype: int64

--- drink ---
drink
False    1126
NaN       804
True      368
Name: count, dtype: int64

--- background_father ---
background_father
NaN            818
POMERANIA      530
GERMANY        466
ITALY          250
UNK             95
BRAZIL          89
NETHERLANDS     19
PORTUGAL        15
POLAND           7
BRASIL           3
CZECH            3
AUSTRIA          1
SPAIN            1
ISRAEL           1
Name: count, dtype: int64

--- background_mother ---
background_mother
NaN            822
POMERANIA      533
GERMANY        482
ITALY          236
UNK             84
BRAZIL          81
NETHERLANDS     20
PORTUGAL        20
POLAND           9
NORWAY           7
FRANCE           2
SPAIN            2
Name: count, dtype: int64

--- age ---
count    2298.00
mean       60.46
std        15.89
min         6.00
25%        52.00
50%        62.00
75%        72.00
max        94.00
Name: age, dtype

In [7]:
#CHUẨN HÓA DỮ LIỆU
# 1. Các cột dạng số
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 2. Các cột dạng chuỗi / phân loại
categorical_columns = [
    "patient_id",
    "background_father",
    "background_mother",
    "gender",
    "region",
    "diagnostic",
    "img_id"
]

for col in categorical_columns:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .str.upper()
    )

# 3. Các cột TRUE/FALSE/UNK
boolean_columns = [
    "smoke", "drink", "pesticide",
    "skin_cancer_history", "cancer_history",
    "has_piped_water", "has_sewage_system",
    "itch", "grew", "hurt", "changed", "bleed", "elevation",
    "biopsed"
]

for col in boolean_columns:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .str.upper()
    )

# 4. Giá trị "UNK" (không biết) thực chất là dữ liệu thiếu -> đổi về NaN
for col in boolean_columns + categorical_columns:
    df[col] = df[col].replace("UNK", np.nan)

print("Đã chuẩn hóa xong dữ liệu.")
print("\nKiểm tra lại giá trị của vài cột:")
print(df["diagnostic"].value_counts())
print(df["itch"].value_counts(dropna=False))

Đã chuẩn hóa xong dữ liệu.

Kiểm tra lại giá trị của vài cột:
diagnostic
BCC    845
ACK    730
NEV    244
SEK    235
SCC    192
MEL     52
Name: count, dtype: Int64
itch
TRUE     1455
FALSE     837
<NA>        6
Name: count, dtype: Int64


In [9]:
#XỬ LÝ DỮ LIỆU THIẾU - TRÙNG:
rows_before = len(df)
print("Số dòng trước khi xử lý:", rows_before)

# Bước 1: Loại bỏ dòng trùng hoàn toàn
df = df.drop_duplicates()
print("Số dòng sau khi bỏ trùng:", len(df))

# Bước 2: Loại bỏ dòng thiếu mã bệnh nhân, mã ảnh hoặc nhãn chẩn đoán
df = df.dropna(subset=["patient_id", "img_id", "diagnostic"])
print("Số dòng sau khi bỏ dòng thiếu thông tin quan trọng:", len(df))

# Bước 3: Đánh lại chỉ số
df = df.reset_index(drop=True)

print("Tổng số dòng đã loại bỏ:", rows_before - len(df))

# XỬ LÝ GIÁ TRỊ BẤT THƯỜNG :
print("KHOẢNG GIÁ TRỊ HIỆN TẠI:")
print(df[numeric_columns].describe().round(2))

# Điều kiện giá trị không hợp lý
age_invalid = (df["age"] < 0) | (df["age"] > 110)
diameter_1_invalid = (df["diameter_1"] <= 0) | (df["diameter_1"] > 100)
diameter_2_invalid = (df["diameter_2"] <= 0) | (df["diameter_2"] > 100)
fitz_invalid = (df["fitspatrick"] < 1) | (df["fitspatrick"] > 6)

print("\nSỐ GIÁ TRỊ BẤT THƯỜNG:")
print("age:", age_invalid.sum())
print("diameter_1:", diameter_1_invalid.sum())
print("diameter_2:", diameter_2_invalid.sum())
print("fitspatrick:", fitz_invalid.sum())

# Đặt các giá trị bất thường thành NaN (sẽ xử lý cùng với dữ liệu thiếu)
df.loc[age_invalid, "age"] = np.nan
df.loc[diameter_1_invalid, "diameter_1"] = np.nan
df.loc[diameter_2_invalid, "diameter_2"] = np.nan
df.loc[fitz_invalid, "fitspatrick"] = np.nan

print("\nĐã đặt các giá trị bất thường thành NaN.")

Số dòng trước khi xử lý: 2298
Số dòng sau khi bỏ trùng: 2298
Số dòng sau khi bỏ dòng thiếu thông tin quan trọng: 2298
Tổng số dòng đã loại bỏ: 0
KHOẢNG GIÁ TRỊ HIỆN TẠI:
           age  fitspatrick  diameter_1  diameter_2
count  2298.00      1494.00     1494.00     1494.00
mean     60.46         2.27       11.90        8.85
std      15.89         0.73        8.63        5.80
min       6.00         1.00        0.00        0.00
25%      52.00         2.00        7.00        5.00
50%      62.00         2.00       10.00        8.00
75%      72.00         3.00       15.00       10.00
max      94.00         6.00      100.00       70.00

SỐ GIÁ TRỊ BẤT THƯỜNG:
age: 0
diameter_1: 4
diameter_2: 4
fitspatrick: 0

Đã đặt các giá trị bất thường thành NaN.


In [10]:
# TẠO BIẾN MỚI :
# Đường kính trung bình của tổn thương
df["AverageDiameter"] = df[["diameter_1", "diameter_2"]].mean(axis=1)

# Nhóm tuổi (dùng cho phân tích, không đưa vào mô hình)
bins = [0, 40, 50, 60, 70, 80, 120]
labels = ["0-40", "41-50", "51-60", "61-70", "71-80", "81+"]

df["AgeGroup"] = pd.cut(
    df["age"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

print("PHÂN BỐ NHÓM TUỔI:")
print(df["AgeGroup"].value_counts().sort_index())

print("\nĐƯỜNG KÍNH TRUNG BÌNH:")
print(df["AverageDiameter"].describe().round(2))

PHÂN BỐ NHÓM TUỔI:
AgeGroup
0-40     260
41-50    261
51-60    556
61-70    551
71-80    475
81+      195
Name: count, dtype: int64

ĐƯỜNG KÍNH TRUNG BÌNH:
count    1490.00
mean       10.40
std         6.95
min         1.00
25%         6.00
50%         9.00
75%        12.50
max        85.00
Name: AverageDiameter, dtype: float64


In [11]:
# CHỌN ĐẶC TRƯNG VÀ ĐIỀN DỮ LIỆU THIẾU
# Cố ý KHÔNG đưa vào mô hình:
#  - biopsed: cho biết đã sinh thiết hay chưa, phụ thuộc kết quả chẩn đoán (rò rỉ nhãn)
#  - patient_id, lesion_id, img_id: chỉ là mã định danh
#  - background_father/mother: quá nhiều giá trị khác nhau và thiếu nhiều
numeric_features = ["age", "diameter_1", "diameter_2", "AverageDiameter"]
categorical_features = [
    "gender", "region", "itch", "grew",
    "hurt", "changed", "bleed", "elevation"
]

meta = df[numeric_features + categorical_features].copy()

print("SỐ DỮ LIỆU THIẾU TRƯỚC KHI ĐIỀN:")
print(meta.isnull().sum())

# Cột số: điền -1 (nghĩa là "không có thông tin")
for col in numeric_features:
    meta[col] = meta[col].fillna(-1)

# Cột chữ: điền "UNK" (nghĩa là "không biết")
for col in categorical_features:
    meta[col] = meta[col].astype("object").fillna("UNK").astype(str)

print("\nSỐ DỮ LIỆU THIẾU SAU KHI ĐIỀN:")
print(meta.isnull().sum().sum())

SỐ DỮ LIỆU THIẾU TRƯỚC KHI ĐIỀN:
age                  0
diameter_1         808
diameter_2         808
AverageDiameter    808
gender             804
region               0
itch                 6
grew               402
hurt                10
changed            396
bleed                6
elevation            2
dtype: int64

SỐ DỮ LIỆU THIẾU SAU KHI ĐIỀN:
0


In [12]:
#MÃ HÓA ONE-HOT:
meta_encoded = pd.get_dummies(meta, columns=categorical_features, dtype=int)

print("Kích thước bảng đặc trưng sau mã hóa:", meta_encoded.shape)
print("\nDanh sách cột:")
print(meta_encoded.columns.tolist())

print("\n5 dòng đầu tiên:")
print(meta_encoded.head())

# Kiểm tra lần cuối
if meta_encoded.isnull().sum().sum() == 0:
    print("\nBảng đặc trưng không còn giá trị thiếu.")
else:
    print("\nCẢNH BÁO: vẫn còn giá trị thiếu!")

Kích thước bảng đặc trưng sau mã hóa: (2298, 39)

Danh sách cột:
['age', 'diameter_1', 'diameter_2', 'AverageDiameter', 'gender_FEMALE', 'gender_MALE', 'gender_UNK', 'region_ABDOMEN', 'region_ARM', 'region_BACK', 'region_CHEST', 'region_EAR', 'region_FACE', 'region_FOOT', 'region_FOREARM', 'region_HAND', 'region_LIP', 'region_NECK', 'region_NOSE', 'region_SCALP', 'region_THIGH', 'itch_FALSE', 'itch_TRUE', 'itch_UNK', 'grew_FALSE', 'grew_TRUE', 'grew_UNK', 'hurt_FALSE', 'hurt_TRUE', 'hurt_UNK', 'changed_FALSE', 'changed_TRUE', 'changed_UNK', 'bleed_FALSE', 'bleed_TRUE', 'bleed_UNK', 'elevation_FALSE', 'elevation_TRUE', 'elevation_UNK']

5 dòng đầu tiên:
    age  diameter_1  diameter_2  AverageDiameter  gender_FEMALE  gender_MALE  \
0   8.0        -1.0        -1.0             -1.0              0            0   
1  55.0         6.0         5.0              5.5              1            0   
2  77.0        -1.0        -1.0             -1.0              0            0   
3  75.0        -1.0

PHẦN 2: TIỀN XỬ LÝ DỮ LIỆU ẢNH